<a href="https://colab.research.google.com/github/Itachi2024/Redrob-AI-Hackathon/blob/main/AI_Recruiter_Colab_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Candidate Discovery and Ranking
## Redrob Hackathon: Senior AI Engineer Search

End-to-end processing: Data Loading, Filtering, Scoring, and Submission generation.

| Parameter | Value |
|-----------|-------|
| Dataset | 100,000 candidates |
| Role | Senior AI Engineer |
| Output | `submission.csv` |
| Runtime | ~3 min |

## 1. Environment Setup

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

# Standard library imports
import json, csv, re, os, math, time
from datetime import datetime
from collections import Counter, defaultdict

# Data science stack
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

print("Environment initialized")

## 2. Data Access
Mount Drive and locate the input file.

In [ ]:
import os

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Google Drive mounted successfully")
except ImportError:
    IN_COLAB = False
    print("Not in Colab — using local mode")

# ================================================================
# EDIT THIS LINE if auto-detect fails:
# CANDIDATES_PATH = '/content/drive/MyDrive/candidates.jsonl'
# ================================================================

POSSIBLE_PATHS = [
    '/content/candidates.jsonl',
    '/content/drive/MyDrive/candidates.jsonl',
    '/content/drive/MyDrive/India Runs/candidates.jsonl',
    '/content/drive/MyDrive/hackathon/candidates.jsonl',
    '/content/drive/MyDrive/Colab Notebooks/candidates.jsonl',
]

CANDIDATES_PATH = None
for p in POSSIBLE_PATHS:
    if os.path.exists(p):
        CANDIDATES_PATH = p
        break

if CANDIDATES_PATH:
    size_mb = os.path.getsize(CANDIDATES_PATH) / (1024 * 1024)
    print(f"Found: {CANDIDATES_PATH} ({size_mb:.1f} MB)")
else:
    print("candidates.jsonl not found in common locations.")
    print("Options:")
    print("  A) Set path manually above and re-run this cell")
    print("  B) Run Step 2b to upload directly")


## 📤 Step 2b: Direct Upload to Colab (Alternative)
**Skip this cell** if Step 2 found the file. Only run if `CANDIDATES_PATH` is still None.


In [ ]:
if CANDIDATES_PATH is None:
    try:
        from google.colab import files
        print("Select candidates.jsonl (487 MB, may take 2-3 min to upload)...")
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith('.jsonl') or 'candidates' in fname.lower():
                CANDIDATES_PATH = f'/content/{fname}'
                print(f"Uploaded to: {CANDIDATES_PATH} ({len(uploaded[fname])/1e6:.1f} MB)")
                break
    except Exception as e:
        print(f"Upload error: {e}")
        print("Please set CANDIDATES_PATH manually in Step 2")
else:
    print(f"Already found: {CANDIDATES_PATH} — skipping upload")


## 3. Configuration

In [ ]:
import json, csv, re, os, sys, math, time
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

# Style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (14, 6)

# Paths
OUTPUT_DIR     = '/content'
SUBMISSION_FILE = os.path.join(OUTPUT_DIR, 'submission.csv')
METADATA_FILE   = os.path.join(OUTPUT_DIR, 'submission_metadata.yaml')

# Reference date for recency calculations
REF_DATE = datetime(2026, 6, 15)

assert CANDIDATES_PATH is not None, "ERROR: Set CANDIDATES_PATH in Step 2 first!"
print("All imports OK")
print(f"Candidates: {CANDIDATES_PATH}")
print(f"Output to : {OUTPUT_DIR}")


## 4. Data Loading and EDA

In [ ]:
print("=" * 65)
print("STAGE 1: Loading candidate data")
print("=" * 65)

pipeline_start = time.time()
stage1_start   = time.time()

candidates = []
file_size  = os.path.getsize(CANDIDATES_PATH)

with open(CANDIDATES_PATH, 'r', encoding='utf-8') as f:
    with tqdm(total=file_size, unit='B', unit_scale=True, desc="Loading") as pbar:
        for line in f:
            pbar.update(len(line.encode('utf-8')))
            line = line.strip()
            if line:
                try:
                    candidates.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

stage1_time = time.time() - stage1_start
print(f"\nLoaded {len(candidates):,} candidates in {stage1_time:.1f}s")
sample = candidates[0]
print(f"  First ID      : {candidates[0]['candidate_id']}")
print(f"  Last  ID      : {candidates[-1]['candidate_id']}")
print(f"  profile keys  : {list(sample.get('profile', {}).keys())}")
print(f"  career count  : {len(sample.get('career_history', []))}")
print(f"  skills count  : {len(sample.get('skills', []))}")
print(f"  signals count : {len(sample.get('redrob_signals', {}))}")


In [ ]:
# Quick EDA — extract flat stats
profiles_data = []
for c in tqdm(candidates, desc="EDA"):
    p = c.get('profile', {})
    s = c.get('redrob_signals', {})
    profiles_data.append({
        'title':        p.get('current_title', ''),
        'yoe':          p.get('years_of_experience', 0),
        'country':      p.get('country', ''),
        'location':     p.get('location', ''),
        'num_skills':   len(c.get('skills', [])),
        'num_careers':  len(c.get('career_history', [])),
        'open_to_work': s.get('open_to_work_flag', False),
        'response_rate':s.get('recruiter_response_rate', 0),
        'notice_days':  s.get('notice_period_days', 0),
        'github_score': s.get('github_activity_score', -1),
    })

df = pd.DataFrame(profiles_data)
print(f"Dataset: {df.shape[0]:,} rows")
print(f"  Countries      : {df['country'].nunique()}")
print(f"  Unique titles  : {df['title'].nunique()}")
print(f"  Avg YoE        : {df['yoe'].mean():.1f} years")
print(f"  Open to work   : {df['open_to_work'].mean()*100:.1f}%")
print(f"  Avg skills     : {df['num_skills'].mean():.1f}")


In [ ]:
# EDA Visualisations (6-panel)
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle("Candidate Dataset Overview — 100K Profiles", fontsize=17, fontweight='bold')

ax = axes[0,0]
ax.hist(df['yoe'], bins=50, color='#4C72B0', alpha=0.85, edgecolor='white')
ax.axvspan(5, 9, alpha=0.13, color='green', label='JD sweet spot (5-9yr)')
ax.axvline(7, color='red', linestyle='--', alpha=0.8, label='Ideal: 7yr')
ax.set_xlabel('Years of Experience'); ax.set_ylabel('Count')
ax.set_title('Experience Distribution', fontweight='bold'); ax.legend(fontsize=8)

ax = axes[0,1]
top_titles = df['title'].value_counts().head(15)
ax.barh(top_titles.index[::-1], top_titles.values[::-1], color='#DD8452')
ax.set_xlabel('Count'); ax.set_title('Top 15 Job Titles', fontweight='bold')

ax = axes[0,2]
top_countries = df['country'].value_counts().head(10)
ax.barh(top_countries.index[::-1], top_countries.values[::-1], color='#55A868')
ax.set_xlabel('Count'); ax.set_title('Top 10 Countries', fontweight='bold')

ax = axes[1,0]
ax.hist(df['num_skills'], bins=30, color='#C44E52', alpha=0.85, edgecolor='white')
ax.set_xlabel('Number of Skills'); ax.set_ylabel('Count')
ax.set_title('Skills per Candidate', fontweight='bold')

ax = axes[1,1]
ax.hist(df['response_rate'], bins=50, color='#8172B3', alpha=0.85, edgecolor='white')
ax.set_xlabel('Recruiter Response Rate')
ax.set_title('Response Rate Distribution', fontweight='bold')

ax = axes[1,2]
ax.hist(df['notice_days'], bins=30, color='#937860', alpha=0.85, edgecolor='white')
ax.axvline(30, color='green', linestyle='--', label='30d (ideal)')
ax.axvline(60, color='orange', linestyle='--', label='60d')
ax.set_xlabel('Notice Period (days)')
ax.set_title('Notice Period Distribution', fontweight='bold'); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()
print("EDA visualisations done")

del df, profiles_data
import gc; gc.collect()


## 5. Skills Distribution

In [ ]:
print("=" * 65)
print("STAGE 2: Skills Landscape")
print("=" * 65)

all_skills = Counter()
all_profs  = Counter()
for c in candidates:
    for s in c.get('skills', []):
        all_skills[s.get('name', '').strip()] += 1
        all_profs[s.get('proficiency', 'unknown')] += 1

print(f"  Unique skills : {len(all_skills):,}")
print(f"  Proficiency   : {dict(all_profs)}")

AI_KW = {'machine learning','deep learning','nlp','pytorch','tensorflow','bert',
         'transformers','embeddings','faiss','pinecone','qdrant','milvus',
         'recommendation','ranking','search','retrieval','lora','fine-tuning',
         'llm','gpt','langchain','rag','computer vision','neural network',
         'keras','scikit-learn','xgboost','data science','ai','python','sql'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
top30 = all_skills.most_common(30)
names, counts = zip(*top30)
colors = ['#e74c3c' if any(kw in n.lower() for kw in AI_KW) else '#3498db' for n in names]
ax1.barh(list(names)[::-1], list(counts)[::-1], color=colors[::-1])
ax1.set_xlabel('Count')
ax1.set_title('Top 30 Skills (red = AI/ML)', fontweight='bold', fontsize=13)

ax2.pie(all_profs.values(), labels=all_profs.keys(), autopct='%1.1f%%',
        colors=['#2ecc71','#3498db','#f39c12','#e74c3c'][:len(all_profs)], startangle=90)
ax2.set_title('Skill Proficiency Distribution', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()
print("Skills analysis done")


## 6. Data Cleaning and Filtering
Filtering out profiles that meet specific heuristic red-flags to improve candidate quality.

In [ ]:
print("=" * 65)
print("STAGE 3: Honeypot Detection")
print("=" * 65)

stage2_start = time.time()

AI_SKILL_KEYWORDS = {
    'machine learning','deep learning','neural network','nlp',
    'natural language processing','computer vision','tensorflow',
    'pytorch','keras','scikit-learn','sklearn','transformers',
    'bert','gpt','llm','large language model','embeddings',
    'vector database','pinecone','weaviate','qdrant','milvus',
    'faiss','retrieval','rag','recommendation','ranking',
    'search','information retrieval','sentence-transformers',
    'fine-tuning','fine tuning','lora','qlora','peft',
    'reinforcement learning','gan','cnn','rnn','lstm','attention',
    'transformer','hugging face','huggingface','opencv',
    'object detection','image classification','speech recognition',
    'xgboost','lightgbm','catboost','gradient boosting','random forest',
    'model deployment','mlops','ml pipeline','feature engineering',
    'data science','ai','artificial intelligence','ndcg','mrr',
    'langchain','llamaindex','openai','chatbot','prompt engineering',
    'statistical modeling','bentoml','wandb',
}

NON_TECH_TITLES = {
    'hr manager','marketing manager','accountant','sales executive',
    'content writer','graphic designer','operations manager',
    'customer support','civil engineer','mechanical engineer',
    'project manager','business analyst','teacher','professor',
    'nurse','doctor','pharmacist','lawyer','chef','receptionist',
    'admin assistant','office manager','event planner',
}

def count_ai_skills(skill_list):
    count = 0
    for s in skill_list:
        nm = s.get('name','').lower().strip()
        if any(kw in nm or nm in kw for kw in AI_SKILL_KEYWORDS):
            count += 1
    return count

def detect_honeypot(cand):
    flags = []
    profile = cand.get('profile', {})
    skills  = cand.get('skills', [])
    career  = cand.get('career_history', [])
    title   = profile.get('current_title', '').lower().strip()
    yoe     = profile.get('years_of_experience', 0)

    # Check 1: expert skills with near-zero duration
    eld = sum(1 for s in skills
              if s.get('proficiency','').lower()=='expert'
              and (s.get('duration_months',0) or 0) < 6)
    if eld >= 3:
        flags.append(f"expert_zero_duration: {eld}x")

    # Check 2: title-skill mismatch
    is_non_tech = any(nt in title for nt in NON_TECH_TITLES)
    ai_count    = count_ai_skills(skills)
    if is_non_tech and ai_count >= 6:
        flags.append(f"title_skill_mismatch: {ai_count} AI skills")

    # Check 3: impossible experience math
    total_months = sum(c.get('duration_months',0) or 0 for c in career)
    total_yrs    = total_months / 12.0
    if yoe > 0 and total_yrs > 0 and yoe > total_yrs * 1.8 + 2:
        flags.append(f"impossible_exp: {yoe:.1f}yr stated vs {total_yrs:.1f}yr career")

    # Check 4: multiple current positions
    if sum(1 for c in career if c.get('is_current',False)) > 2:
        flags.append("multi_current")

    # Check 5: absurd endorsements
    ae = sum(1 for s in skills
             if (s.get('duration_months',0) or 0) <= 3
             and (s.get('endorsements',0) or 0) >= 40)
    if ae >= 3:
        flags.append(f"absurd_endorsements: {ae}x")

    # Check 6: identical career descriptions
    if len(career) >= 3:
        descs = [c.get('description','').strip() for c in career]
        non_empty = [d for d in descs if d]
        if len(non_empty) >= 3 and len(set(non_empty)) == 1:
            flags.append("identical_descriptions")

    # Check 7: no tech in career despite AI skills
    if is_non_tech and ai_count >= 5:
        all_desc = ' '.join(c.get('description','') for c in career).lower()
        tech_hits = sum(1 for kw in ['machine learning','deep learning','model',
                                      'neural','algorithm','data science','ml',
                                      'embedding','vector','retrieval','nlp',
                                      'recommendation','ranking','pytorch',
                                      'tensorflow','training'] if kw in all_desc)
        if tech_hits <= 2:
            flags.append("no_tech_in_career")
    return flags

honeypot_flags   = {}
clean_candidates = []
for cand in tqdm(candidates, desc="Honeypot scan"):
    cid   = cand['candidate_id']
    flags = detect_honeypot(cand)
    if flags:
        honeypot_flags[cid] = flags
    else:
        clean_candidates.append(cand)

n_honeypots = len(honeypot_flags)
n_clean     = len(clean_candidates)
stage2_time = time.time() - stage2_start

print(f"\nHoneypot detection done in {stage2_time:.1f}s")
print(f"  Honeypots flagged : {n_honeypots:,}")
print(f"  Clean candidates  : {n_clean:,}")
print(f"  Honeypot rate     : {n_honeypots/len(candidates)*100:.2f}%")

hp_reasons = Counter(f.split(':')[0] for flags in honeypot_flags.values() for f in flags)
print("\nFlag breakdown:")
for reason, cnt in hp_reasons.most_common():
    print(f"  {reason}: {cnt}")

print("\nSample honeypots (first 5):")
for cid, flags in list(honeypot_flags.items())[:5]:
    c = next(x for x in candidates if x['candidate_id']==cid)
    t = c.get('profile',{}).get('current_title','?')
    print(f"  {cid}: {t}")
    for f in flags: print(f"    -> {f}")


## 7. Semantic Matching
Using TF-IDF and Cosine Similarity to compare candidate profiles against the job description.

In [ ]:
print("=" * 65)
print("STAGE 4: JD Parsing + TF-IDF Semantic Matching")
print("=" * 65)

stage4_start = time.time()

JD_TEXT = """
Senior AI Engineer at Redrob AI. Location: Pune or Noida, India. Hybrid work mode.
Experience required: 5 to 9 years, flexible for exceptional candidates.

We are building the AI backbone of Redrob - the intelligent layer that understands jobs,
understands candidates, and matches them better than any keyword search can.
We need someone who has shipped production embedding and retrieval systems.

Must-have technical skills:
- Production experience with embedding systems and retrieval pipelines
- Vector databases: Pinecone, Weaviate, Qdrant, Milvus, or FAISS
- Strong Python programming for production-grade systems
- Ranking evaluation metrics: NDCG, MRR, MAP, precision at K
- Search, ranking, or recommendation systems shipped to production
- Applied NLP and machine learning in production
- Deep learning frameworks: PyTorch or TensorFlow
- Sentence transformers, semantic search, similarity matching

Nice-to-have:
- LLM fine-tuning with LoRA, QLoRA, PEFT
- Learning-to-rank algorithms
- HR-tech domain knowledge, recruiting, talent matching
- Distributed systems at scale
- Open-source contributions
- XGBoost, gradient boosting for ranking

Ideal candidate: 6-8 years total, 4-5 in applied ML/AI at product companies,
has shipped a ranking, search, or recommendation system to production.
Product company background preferred over consulting/services.
"""

CORE_SKILLS = [
    'embeddings','embedding','retrieval','vector database','vector db',
    'pinecone','weaviate','qdrant','milvus','faiss',
    'elasticsearch','opensearch','sentence-transformers','sbert',
    'python','ndcg','mrr','map','precision',
    'ranking','search','nlp','natural language processing',
    'information retrieval','recommendation','recommender',
    'pytorch','torch','tensorflow','deep learning','machine learning',
    'ml','ai','neural network','transformer','bert','gpt',
    'semantic search','similarity','production','deployed','shipped',
]

NICE_SKILLS = [
    'lora','qlora','peft','fine-tuning','fine tuning','finetuning',
    'llm','large language model','distributed systems',
    'xgboost','lightgbm','gradient boosting','catboost',
    'learning to rank','learning-to-rank','hr-tech','hr tech',
    'recruiting','talent','open source','open-source',
    'kubernetes','docker','kafka','spark',
]

IDEAL_YOE_CENTER = 7.0
IDEAL_YOE_SIGMA  = 2.0
YOE_MIN          = 5.0
YOE_MAX          = 9.0
SALARY_MIN_LPA   = 25.0
SALARY_MAX_LPA   = 50.0
PREFERRED_LOCATIONS = {
    'pune':1.0,'noida':1.0,
    'delhi':0.85,'new delhi':0.85,'gurgaon':0.85,'gurugram':0.85,
    'delhi ncr':0.85,'ncr':0.85,'ghaziabad':0.85,'faridabad':0.85,
    'bangalore':0.8,'bengaluru':0.8,'mumbai':0.8,'hyderabad':0.8,
    'chennai':0.7,'kolkata':0.7,
}

print(f"  Core skills  : {len(CORE_SKILLS)}")
print(f"  Nice-to-have : {len(NICE_SKILLS)}")
print(f"  Ideal YoE    : {IDEAL_YOE_CENTER} +/- {IDEAL_YOE_SIGMA} years")
print(f"  Salary range : Rs {SALARY_MIN_LPA:.0f}-{SALARY_MAX_LPA:.0f} LPA")

# Build candidate text representations
def build_candidate_text(cand):
    parts = []
    p = cand.get('profile', {})
    parts += [p.get('headline',''), p.get('summary',''),
              p.get('current_title',''), p.get('current_company',''),
              p.get('current_industry','')]
    for j in cand.get('career_history', []):
        parts += [j.get('title',''), j.get('description',''), j.get('industry','')]
    parts.append(' '.join(s.get('name','') for s in cand.get('skills', [])))
    parts.append(' '.join(c.get('name','') for c in cand.get('certifications', [])))
    for e in cand.get('education', []):
        parts += [e.get('field_of_study',''), e.get('degree','')]
    return ' '.join(x for x in parts if x).strip()

candidate_texts = []
candidate_ids_ordered = []
for cand in tqdm(clean_candidates, desc="Building texts"):
    candidate_texts.append(build_candidate_text(cand))
    candidate_ids_ordered.append(cand['candidate_id'])

print(f"  Built {len(candidate_texts):,} text docs")
print(f"  Avg text length: {sum(len(t) for t in candidate_texts)/len(candidate_texts):.0f} chars")

# TF-IDF fit + cosine similarity
print("  Fitting TF-IDF vectorizer...")
all_texts = [JD_TEXT] + candidate_texts
tfidf = TfidfVectorizer(
    ngram_range=(1,2), max_features=10000, sublinear_tf=True,
    stop_words='english', min_df=2, max_df=0.95, dtype=np.float32)
tfidf_matrix = tfidf.fit_transform(all_texts)
print(f"  Vocab: {len(tfidf.vocabulary_):,} terms | Matrix: {tfidf_matrix.shape}")

print("  Computing cosine similarities...")
semantic_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
smin, smax = semantic_scores.min(), semantic_scores.max()
semantic_scores_norm = ((semantic_scores-smin)/(smax-smin)
                        if smax > smin else np.zeros_like(semantic_scores))

stage4_time = time.time() - stage4_start
print(f"\nSemantic matching done in {stage4_time:.1f}s")
print(f"  Score range: [{smin:.4f}, {smax:.4f}] | Mean: {semantic_scores.mean():.4f}")

del all_texts, tfidf_matrix
import gc; gc.collect()


## 8. Feature Scoring
Calculating weighted scores across multiple dimensions including experience, location, and technical skills.

In [ ]:
print("=" * 65)
print("STAGE 5: Multi-Dimensional Feature Scoring")
print("=" * 65)

stage5_start = time.time()

def match_skills_list(skill_names_lower, keyword_list):
    matched = 0
    for kw in keyword_list:
        kl = kw.lower()
        for sn in skill_names_lower:
            if kl in sn or sn in kl:
                matched += 1; break
    return matched

PROFICIENCY_WEIGHT = {'expert':1.0,'advanced':0.8,'intermediate':0.5,'beginner':0.3}

TITLE_RELEVANCE = {
    'ai engineer':1.0,'senior ai engineer':1.0,'ml engineer':1.0,
    'machine learning engineer':1.0,'senior ml engineer':1.0,
    'senior machine learning engineer':1.0,'staff ml engineer':1.0,
    'principal ml engineer':1.0,'data scientist':0.85,
    'senior data scientist':0.9,'lead data scientist':0.9,
    'principal data scientist':0.9,'research scientist':0.7,
    'research engineer':0.75,'applied scientist':0.85,'nlp engineer':0.9,
    'search engineer':0.9,'ranking engineer':0.95,
    'recommendation engineer':0.9,'retrieval engineer':0.95,
    'software engineer':0.6,'senior software engineer':0.65,
    'backend engineer':0.5,'data engineer':0.55,
    'full stack engineer':0.4,'devops engineer':0.3,
    'junior ml engineer':0.7,'junior data scientist':0.5,
    'data analyst':0.4,'business analyst':0.15,
    'project manager':0.1,'product manager':0.2,
    'hr manager':0.05,'marketing manager':0.05,'accountant':0.02,
    'sales executive':0.05,'content writer':0.05,'graphic designer':0.03,
    'operations manager':0.05,'customer support':0.03,
    'civil engineer':0.05,'mechanical engineer':0.05,
    'teacher':0.05,'professor':0.15,
}

PRODUCTION_KEYWORDS = [
    'production','deployed','deployment','shipped','launched',
    'released','live','scale','scaling','served','serving',
    'api','microservice','pipeline','real-time','realtime',
    'monitoring','a/b test','ab test','latency','throughput',
    'sla','uptime','kubernetes','docker','cicd','ci/cd',
    'millions of users','thousands of requests','traffic',
    'infrastructure','production-grade','end-to-end',
]

feature_names = [
    'skills_match','semantic','experience_fit','title_alignment',
    'education','location','notice','salary_fit',
    'engagement','market_demand','production_exp','anti_signal_penalty'
]

all_features = np.zeros((len(clean_candidates), len(feature_names)), dtype=np.float32)

def safe_parse_date(ds):
    if not ds: return None
    try: return datetime.strptime(str(ds)[:10], '%Y-%m-%d')
    except: return None

for idx, cand in enumerate(tqdm(clean_candidates, desc="Scoring features")):
    profile  = cand.get('profile', {})
    skills   = cand.get('skills', [])
    career   = cand.get('career_history', [])
    edu_list = cand.get('education', [])
    certs    = cand.get('certifications', [])
    signals  = cand.get('redrob_signals', {})

    title   = profile.get('current_title','').lower().strip()
    yoe     = profile.get('years_of_experience', 0) or 0
    location= profile.get('location','').lower().strip()
    country = profile.get('country','').lower().strip()
    all_career_text = ' '.join(
        j.get('description','') + ' ' + j.get('title','') for j in career).lower()
    skill_names_lower = [s.get('name','').lower().strip() for s in skills]

    # a) skills_match
    core_weighted = 0.0
    for kw in CORE_SKILLS:
        kl = kw.lower()
        for s in skills:
            sn = s.get('name','').lower().strip()
            if kl in sn or sn in kl:
                pw  = PROFICIENCY_WEIGHT.get(s.get('proficiency','').lower(), 0.3)
                dur = s.get('duration_months', 0) or 0
                core_weighted += pw * (0.6 + 0.4*min(dur/36.0, 1.0))
                break
    career_core_bonus = min(sum(1 for kw in CORE_SKILLS if kw.lower() in all_career_text)/10.0, 0.3)
    nice_bonus = min(match_skills_list(skill_names_lower, NICE_SKILLS)/len(NICE_SKILLS), 0.2)
    skills_score = min(core_weighted/12.0 + career_core_bonus + nice_bonus, 1.0)

    # b) semantic (pre-computed)
    semantic_s = float(semantic_scores_norm[idx])

    # c) experience_fit
    exp_gaussian  = math.exp(-0.5*((yoe-IDEAL_YOE_CENTER)/IDEAL_YOE_SIGMA)**2)
    industries    = [j.get('industry','').lower() for j in career]
    product_exp   = sum(1 for ind in industries if any(pk in ind for pk in
                        ['product','startup','fintech','edtech','healthtech',
                         'saas','platform','e-commerce','ecommerce','gaming']))
    consulting_exp= sum(1 for ind in industries if any(ck in ind for ck in
                        ['consulting','staffing','outsourcing','bpo']))
    exp_score = min(max(exp_gaussian + min(product_exp*0.1,0.15) - min(consulting_exp*0.05,0.1), 0), 1.0)

    # d) title_alignment
    title_score = 0.1
    for tk, tv in TITLE_RELEVANCE.items():
        if tk in title or title in tk:
            title_score = max(title_score, tv); break
    if any(sk in title for sk in ['senior','lead','staff','principal','head','director','vp']) and title_score>=0.5:
        title_score = min(title_score+0.1, 1.0)
    tech_career = sum(1 for t in [j.get('title','').lower() for j in career]
                      if any(kw in t for kw in ['engineer','scientist','developer','ml','ai',
                                                 'data','research','architect','nlp','search']))
    if tech_career >= 2: title_score = min(title_score+0.05, 1.0)

    # e) education
    edu_score = 0.3
    if edu_list:
        tier_map = {'tier_1':1.0,'tier_2':0.75,'tier_3':0.5,'tier_4':0.3,'unknown':0.4}
        bt, bf, bd = 0.4, 0.3, 0.5
        for edu in edu_list:
            bt = max(bt, tier_map.get(edu.get('tier','unknown'), 0.4))
            field = edu.get('field_of_study','').lower()
            if any(rf in field for rf in ['computer science','machine learning','artificial intelligence',
                                           'data science','statistics','mathematics','information technology',
                                           'computational','nlp','deep learning']):
                bf = max(bf, 1.0)
            elif any(ef in field for ef in ['engineering','electronics','electrical','ece','eee']):
                bf = max(bf, 0.6)
            degree = edu.get('degree','').lower()
            if 'ph' in degree or 'doctor' in degree: bd = max(bd, 1.0)
            elif any(x in degree for x in ['m.','master','mba','mtech','m.tech']): bd = max(bd, 0.8)
            elif any(x in degree for x in ['b.','bachelor','btech','b.tech']): bd = max(bd, 0.5)
        cert_bonus = sum(0.03 for cert in certs
                         if any(rc in cert.get('name','').lower()
                                for rc in ['aws','gcp','azure','tensorflow','pytorch',
                                           'deep learning','machine learning','data science',
                                           'kubernetes','docker']))
        edu_score = min(bt*0.4 + bf*0.35 + bd*0.25 + cert_bonus, 1.0)

    # f) location
    loc_score = 0.1
    if country == 'india':
        loc_score = 0.5
        for lk, lv in PREFERRED_LOCATIONS.items():
            if lk in location:
                loc_score = max(loc_score, lv); break
    elif signals.get('willing_to_relocate', False):
        loc_score = 0.4
    if signals.get('preferred_work_mode','') in ('hybrid','flexible') and loc_score>=0.5:
        loc_score = min(loc_score+0.05, 1.0)

    # g) notice
    nd = signals.get('notice_period_days', 90) or 90
    notice_score = 1.0 if nd<=30 else 0.7 if nd<=60 else 0.4 if nd<=90 else 0.2

    # h) salary_fit
    sal = signals.get('expected_salary_range_inr_lpa', {})
    sc_min = sal.get('min', 0) or 0
    sc_max = sal.get('max', 0) or 0
    if sc_min == 0 and sc_max == 0:
        salary_score = 0.5
    else:
        ov_s = max(sc_min, SALARY_MIN_LPA)
        ov_e = min(sc_max, SALARY_MAX_LPA)
        if ov_s <= ov_e:
            salary_score = 0.6 + 0.4*min((ov_e-ov_s)/max(sc_max-sc_min,1), 1.0)
        elif sc_max < SALARY_MIN_LPA:
            salary_score = max(0.5-(SALARY_MIN_LPA-sc_max)/20.0, 0.2)
        else:
            salary_score = max(0.3-(sc_min-SALARY_MAX_LPA)/30.0, 0.1)

    # i) engagement
    rr  = signals.get('recruiter_response_rate', 0) or 0
    art = signals.get('avg_response_time_hours', 168) or 168
    otw = 1.0 if signals.get('open_to_work_flag', False) else 0.0
    la  = safe_parse_date(signals.get('last_active_date'))
    recency = math.exp(-(REF_DATE-la).days/180.0) if la else 0.3
    gh  = signals.get('github_activity_score', -1)
    gh_norm = 0.3 if gh < 0 else min(gh/100.0, 1.0)
    icr = signals.get('interview_completion_rate', 0.5) or 0.5
    verification = ((1.0 if signals.get('verified_email') else 0.0) +
                    (1.0 if signals.get('verified_phone') else 0.0) +
                    (1.0 if signals.get('linkedin_connected') else 0.0)) / 3.0
    pcs = (signals.get('profile_completeness_score', 50) or 50) / 100.0
    engagement_score = (rr*0.30 + (1.0-min(art/168.0,1.0))*0.15 + otw*0.15 +
                        recency*0.15 + gh_norm*0.10 + icr*0.05 + verification*0.05 + pcs*0.05)

    # j) market_demand
    saved    = signals.get('saved_by_recruiters_30d', 0) or 0
    views    = signals.get('profile_views_received_30d', 0) or 0
    searches = signals.get('search_appearance_30d', 0) or 0
    market_score = min(
        min(math.log1p(saved)/math.log1p(50), 1.0)*0.4 +
        min(math.log1p(views)/math.log1p(100), 1.0)*0.3 +
        min(math.log1p(searches)/math.log1p(500), 1.0)*0.3, 1.0)

    # k) production_experience
    prod_hits  = sum(1 for kw in PRODUCTION_KEYWORDS if kw in all_career_text)
    prod_score = min(prod_hits/8.0, 1.0)

    # l) anti_signal (multiplicative penalty)
    anti_penalty = 1.0
    if consulting_exp > 0 and product_exp == 0:
        it_svc = sum(1 for ind in industries if 'it services' in ind or 'consulting' in ind)
        if it_svc == len(industries) and len(industries) >= 2:
            anti_penalty *= 0.6
    if len(career) >= 3:
        short_stints = sum(1 for j in career
                           if (j.get('duration_months',0) or 0) < 12
                           and not j.get('is_current', False))
        if short_stints >= 3: anti_penalty *= 0.75
    if prod_hits == 0 and title_score >= 0.5:
        anti_penalty *= 0.8
    cv_speech = sum(1 for sn in skill_names_lower
                    if any(kw in sn for kw in ['speech','tts','asr','computer vision',
                                                'cv','image','object detection']))
    broader_nlp = sum(1 for sn in skill_names_lower
                      if any(kw in sn for kw in ['nlp','search','retrieval','ranking',
                                                   'recommendation','embedding','transformer',
                                                   'bert','text']))
    if cv_speech >= 3 and broader_nlp == 0: anti_penalty *= 0.75

    all_features[idx] = [
        skills_score, semantic_s, exp_score, title_score,
        edu_score, loc_score, notice_score, salary_score,
        engagement_score, market_score, prod_score, anti_penalty
    ]

stage5_time = time.time() - stage5_start
print(f"\nFeature scoring done in {stage5_time:.1f}s")
feature_df = pd.DataFrame(all_features, columns=feature_names)
print("\nFeature Statistics:")
print(feature_df.describe().round(3).to_string())


## 9. Ranking and Final Selection

In [ ]:
print("=" * 65)
print("STAGE 6: Composite Scoring + Top-100 Selection")
print("=" * 65)

WEIGHTS = {
    'skills_match':0.22, 'semantic':0.18, 'experience_fit':0.15,
    'title_alignment':0.12, 'engagement':0.08, 'education':0.06,
    'location':0.05, 'notice':0.04, 'salary_fit':0.04,
    'market_demand':0.03, 'production_exp':0.03,
}
assert abs(sum(WEIGHTS.values())-1.0)<0.01
print(f"Weight sum: {sum(WEIGHTS.values()):.2f} (OK)")

# Weighted composite score x anti-signal multiplier
composite = np.zeros(len(clean_candidates), dtype=np.float64)
for i, fn in enumerate(feature_names[:-1]):
    composite += WEIGHTS[fn] * all_features[:, i].astype(np.float64)
composite *= all_features[:, -1].astype(np.float64)

# Normalize to [0, 1]
cmin, cmax = composite.min(), composite.max()
normalized_scores = ((composite-cmin)/(cmax-cmin)
                     if cmax > cmin else np.zeros_like(composite))

print(f"  Raw range        : [{cmin:.6f}, {cmax:.6f}]")
print(f"  Normalized range : [{normalized_scores.min():.6f}, {normalized_scores.max():.6f}]")

# Build, sort, take top-100
results = [{'candidate_id': cand['candidate_id'],
            'raw_score':    normalized_scores[i],
            'features':     all_features[i].copy(),
            'candidate':    cand}
           for i, cand in enumerate(clean_candidates)]

results.sort(key=lambda x: (-x['raw_score'], x['candidate_id']))
top_100 = results[:100]

# Calibrate: non-increasing, rounded to 4dp
calibrated_scores = []
for i, r in enumerate(top_100):
    sc = round(r['raw_score'], 4)
    if i > 0 and sc > calibrated_scores[-1]:
        sc = calibrated_scores[-1]
    calibrated_scores.append(sc)

# Tie-breaking: equal scores -> candidate_id ascending
i = 0
while i < len(top_100):
    j = i
    while j < len(top_100) and calibrated_scores[j] == calibrated_scores[i]:
        j += 1
    if j-i > 1:
        grp = sorted(range(i,j), key=lambda k: top_100[k]['candidate_id'])
        reordered = [top_100[k] for k in grp]
        for k_idx, k in enumerate(range(i,j)):
            top_100[k] = reordered[k_idx]
    i = j

for rank, entry in enumerate(top_100, 1):
    entry['rank']        = rank
    entry['final_score'] = calibrated_scores[rank-1]

# Verify invariants
for i in range(len(calibrated_scores)-1):
    assert calibrated_scores[i] >= calibrated_scores[i+1]
for i in range(len(top_100)-1):
    if top_100[i]['final_score'] == top_100[i+1]['final_score']:
        assert top_100[i]['candidate_id'] < top_100[i+1]['candidate_id']

print(f"\nTop-100 selected & verified")
print(f"  Score range: [{calibrated_scores[-1]:.4f}, {calibrated_scores[0]:.4f}]")
print(f"\n{'Rank':<6} {'Candidate ID':<16} {'Score':<8} {'Title':<35} {'YoE':<5}")
print("-"*70)
for e in top_100[:10]:
    p = e['candidate'].get('profile',{})
    t = p.get('current_title','N/A')[:33]
    y = p.get('years_of_experience',0)
    print(f"{e['rank']:<6} {e['candidate_id']:<16} {e['final_score']:<8.4f} {t:<35} {y:<5.1f}")


## 10. Reasoning Generation
Generating descriptive summaries for the top-ranked candidates.

In [ ]:
print("=" * 65)
print("STAGE 7: Reasoning Generation")
print("=" * 65)

def generate_reasoning(entry):
    cand    = entry['candidate']
    profile = cand.get('profile',{})
    skills  = cand.get('skills',[])
    career  = cand.get('career_history',[])
    signals = cand.get('redrob_signals',{})

    title   = profile.get('current_title','Unknown')
    company = profile.get('current_company','Unknown')
    yoe     = profile.get('years_of_experience',0)
    location= profile.get('location','')

    sn_lower    = [s.get('name','').lower() for s in skills]
    skill_names = [s.get('name','') for s in skills]

    matched_core = []
    for kw in CORE_SKILLS:
        kl = kw.lower()
        for i, sn in enumerate(sn_lower):
            if kl in sn or sn in kl:
                matched_core.append(skill_names[i]); break
    matched_core = list(dict.fromkeys(matched_core))[:5]

    matched_nice = []
    for kw in NICE_SKILLS:
        kl = kw.lower()
        for i, sn in enumerate(sn_lower):
            if kl in sn or sn in kl:
                matched_nice.append(skill_names[i]); break
    matched_nice = list(dict.fromkeys(matched_nice))[:3]

    act = ' '.join(j.get('description','') for j in career).lower()
    highlights = [label for kw, label in [
        ('production','prod deployment'),('embedding','embeddings'),
        ('retrieval','retrieval'),('ranking','ranking'),
        ('recommendation','rec systems'),('search','search'),
        ('pipeline','ML pipelines'),('scale','scale'),
        ('vector','vector search'),
    ] if kw in act][:2]

    parts = [f"{title} | {yoe:.0f}yr | {company}"]
    if matched_core: parts.append("skills:" + ",".join(matched_core[:3]))
    if highlights:   parts.append("career:" + "; ".join(highlights))
    if matched_nice: parts.append("+:" + ",".join(matched_nice[:2]))

    concerns = []
    nd = signals.get('notice_period_days', 0)
    if nd and nd > 60: concerns.append(f"{nd}d notice")
    rr = signals.get('recruiter_response_rate', 0)
    if rr and rr < 0.3: concerns.append(f"low resp({rr:.0%})")
    la = safe_parse_date(signals.get('last_active_date'))
    if la and (REF_DATE-la).days > 180: concerns.append(f"inactive {(REF_DATE-la).days}d")
    if profile.get('country','').lower() != 'india': concerns.append(f"loc:{location}")
    if yoe < 4: concerns.append("limited exp")
    elif yoe > 10: concerns.append(f"overqualified({yoe:.0f}yr)")
    if concerns: parts.append("concern:" + "; ".join(concerns[:2]))

    r = "; ".join(parts)
    return r[:197]+"..." if len(r) > 200 else r

for entry in tqdm(top_100, desc="Generating reasoning"):
    entry['reasoning'] = generate_reasoning(entry)

reasoning_lens = [len(e['reasoning']) for e in top_100]
print(f"\nReasoning lengths: min={min(reasoning_lens)}, max={max(reasoning_lens)}, "
      f"mean={sum(reasoning_lens)/len(reasoning_lens):.0f}")
print(f"Unique reasonings: {len(set(e['reasoning'] for e in top_100))}/100")

print("\nSample reasoning (Top 5):")
for e in top_100[:5]:
    print(f"  #{e['rank']}: {e['reasoning']}")


## 11. Results Visualization

In [ ]:
print("=" * 65)
print("STAGE 8: Results Visualization")
print("=" * 65)

fig = plt.figure(figsize=(20, 22))
gs  = gridspec.GridSpec(4, 2, hspace=0.38, wspace=0.3)

# 1: Score distribution all candidates
ax1 = fig.add_subplot(gs[0,0])
ax1.hist(normalized_scores, bins=100, color='steelblue', alpha=0.7, edgecolor='white')
ax1.axvline(calibrated_scores[-1], color='red', linestyle='--', linewidth=2,
            label=f"Top-100 cutoff ({calibrated_scores[-1]:.4f})")
ax1.set_xlabel('Normalized Score'); ax1.set_ylabel('Candidates')
ax1.set_title('Score Distribution — All Candidates', fontweight='bold'); ax1.legend()

# 2: Top-100 scores by rank
ax2 = fig.add_subplot(gs[0,1])
ax2.bar(range(1,101), [e['final_score'] for e in top_100], color='coral', alpha=0.8)
ax2.set_xlabel('Rank'); ax2.set_ylabel('Score')
ax2.set_title('Top-100 Score by Rank', fontweight='bold')

# 3: Feature weights
ax3 = fig.add_subplot(gs[1,0])
wk, wv = list(WEIGHTS.keys()), list(WEIGHTS.values())
bars = ax3.barh(wk, wv, color=sns.color_palette('viridis', len(wk)), alpha=0.85)
for bar, val in zip(bars, wv):
    ax3.text(bar.get_width()+0.003, bar.get_y()+bar.get_height()/2,
             f"{val:.2f}", va='center', fontsize=9)
ax3.set_title('Feature Weights in Composite Score', fontweight='bold')

# 4: Top-10 vs Bottom-10 features
ax4 = fig.add_subplot(gs[1,1])
top10f = np.mean([e['features'][:-1] for e in top_100[:10]], axis=0)
bot10f = np.mean([e['features'][:-1] for e in top_100[-10:]], axis=0)
x_pos  = np.arange(len(feature_names)-1)
ax4.bar(x_pos-0.18, top10f, 0.35, label='Top-10', color='gold', alpha=0.8)
ax4.bar(x_pos+0.18, bot10f, 0.35, label='Bottom-10', color='lightcoral', alpha=0.8)
ax4.set_xticks(x_pos); ax4.set_xticklabels(feature_names[:-1], rotation=45, ha='right', fontsize=8)
ax4.set_title('Feature Scores: Top-10 vs Bottom-10', fontweight='bold'); ax4.legend()

# 5: Honeypot flag breakdown
ax5 = fig.add_subplot(gs[2,0])
hp_reasons = Counter(f.split(':')[0] for flags in honeypot_flags.values() for f in flags)
if hp_reasons:
    reasons, counts = zip(*hp_reasons.most_common())
    ax5.barh(list(reasons), list(counts), color='tomato', alpha=0.8)
ax5.set_title(f'Honeypot Flags ({n_honeypots} total)', fontweight='bold')

# 6: Job titles in Top-100
ax6 = fig.add_subplot(gs[2,1])
titles_ctr = Counter(e['candidate'].get('profile',{}).get('current_title','?')[:30] for e in top_100)
tl, tc = zip(*titles_ctr.most_common(12))
ax6.barh(list(tl), list(tc), color='mediumpurple', alpha=0.8)
ax6.set_title('Job Titles in Top-100', fontweight='bold')

# 7: Experience distribution
ax7 = fig.add_subplot(gs[3,0])
top100_yoe = [e['candidate'].get('profile',{}).get('years_of_experience',0) for e in top_100]
ax7.hist(top100_yoe, bins=20, color='seagreen', alpha=0.7, edgecolor='white')
ax7.axvline(IDEAL_YOE_CENTER, color='red', linestyle='--', label=f'Ideal: {IDEAL_YOE_CENTER}yr')
ax7.axvspan(YOE_MIN, YOE_MAX, alpha=0.12, color='green', label=f'Sweet spot')
ax7.set_xlabel('Years of Experience')
ax7.set_title('Experience Distribution — Top-100', fontweight='bold'); ax7.legend()

# 8: Locations in Top-100
ax8 = fig.add_subplot(gs[3,1])
locs_ctr = Counter()
for e in top_100:
    loc = e['candidate'].get('profile',{}).get('location','Unknown')
    locs_ctr[loc.split(',')[0].strip()[:18]] += 1
ll, lc = zip(*locs_ctr.most_common(12))
ax8.barh(list(ll), list(lc), color='darkorange', alpha=0.8)
ax8.set_title('Locations — Top-100', fontweight='bold')

plt.savefig(os.path.join(OUTPUT_DIR, 'ranking_analysis.png'),
            dpi=130, bbox_inches='tight', facecolor='white')
plt.show()
print("Visualization saved to /content/ranking_analysis.png")


## 12. Export Submission Files

In [ ]:
print("=" * 65)
print("STAGE 9: Submission Generation")
print("=" * 65)

# Write submission.csv
with open(SUBMISSION_FILE, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['candidate_id','rank','score','reasoning'])
    for e in top_100:
        w.writerow([e['candidate_id'], e['rank'], f"{e['final_score']:.4f}", e['reasoning']])

# Self-validate
with open(SUBMISSION_FILE, 'r', encoding='utf-8') as f:
    rd  = csv.reader(f)
    hdr = next(rd)
    rows = list(rd)

assert hdr == ['candidate_id','rank','score','reasoning'], f"Bad header: {hdr}"
assert len(rows) == 100, f"Expected 100 rows, got {len(rows)}"
cids  = [r[0] for r in rows]
assert len(set(cids)) == 100, "Duplicate candidate IDs!"
ranks = [int(r[1]) for r in rows]
assert sorted(ranks) == list(range(1,101)), "Ranks not 1-100!"
scores = [float(r[2]) for r in rows]
for i in range(len(scores)-1):
    assert scores[i] >= scores[i+1], f"Score not non-increasing at rank {i+1}"
for i in range(len(rows)-1):
    if float(rows[i][2]) == float(rows[i+1][2]):
        assert rows[i][0] < rows[i+1][0], f"Tie-break violated at rank {i+1}"

pipeline_total = time.time() - pipeline_start
print(f"submission.csv: {os.path.getsize(SUBMISSION_FILE):,} bytes")
print("All validation checks PASSED")
print(f"  Header  : {hdr}")
print(f"  Rows    : {len(rows)} | Score range [{scores[-1]:.4f}, {scores[0]:.4f}]")

# Write metadata YAML
import sys as _sys
meta_content = f"""# Redrob Hackathon - Submission Metadata
team_name: "ai-recruiter-poc"
primary_contact:
  name: "YOUR_NAME"
  email: "your@email.com"
  phone: "+91-XXXXXXXXXX"
team_members:
  - name: "YOUR_NAME"
    email: "your@email.com"
    role: "ML Engineer"
github_repo: "https://github.com/YOUR_USERNAME/ai-recruiter-poc"
sandbox_link: "PASTE_YOUR_COLAB_SHARE_LINK_HERE"
reproduce_command: "Run all cells in AI_Recruiter_Colab_Complete.ipynb"
compute:
  platform: "Google Colab (Free Tier)"
  cpu_cores: {os.cpu_count()}
  ram_gb: 12
  python_version: "{_sys.version.split()[0]}"
  os: "Linux (Google Colab)"
  uses_gpu_for_inference: false
  has_network_during_ranking: false
  pre_computation_required: false
ai_tools_used: ["Claude", "Gemini"]
ai_usage_summary: |
  Used AI for architecture design and code review.
  All ranking logic is deterministic and runs locally.
methodology_summary: |
  Complete pipeline: honeypot detection (7 heuristics) + TF-IDF bigram
  semantic matching + 11-feature weighted scoring (skills, experience,
  title, education, location, engagement, production keywords) with
  multiplicative anti-signal penalty. Top-100 from {n_clean:,} clean candidates.
  Total runtime: {pipeline_total:.0f}s.
performance:
  total_candidates: {len(candidates)}
  honeypots_detected: {n_honeypots}
  clean_candidates: {n_clean}
  runtime_seconds: {pipeline_total:.1f}
declarations:
  read_submission_spec: true
  code_is_original_work: true
  no_collusion: true
  honeypot_check_done: true
  reproduction_tested: true
"""
with open(METADATA_FILE, 'w', encoding='utf-8') as f:
    f.write(meta_content)

print(f"\nPIPELINE COMPLETE in {pipeline_total:.1f}s ({pipeline_total/60:.1f} min)")
print(f"  Candidates : {len(candidates):,}")
print(f"  Honeypots  : {n_honeypots:,} ({n_honeypots/len(candidates)*100:.1f}%)")
print(f"  Runtime    : {'PASS' if pipeline_total<300 else 'WARNING'} (limit: 300s)")

# Auto-download (Colab only)
try:
    from google.colab import files
    print("\nDownloading files...")
    files.download(SUBMISSION_FILE)
    files.download(METADATA_FILE)
    print("Files downloaded to your computer!")
except ImportError:
    print(f"\nFiles saved locally:")
    print(f"  {SUBMISSION_FILE}")
    print(f"  {METADATA_FILE}")


## 13. Preview and Final Checks

In [ ]:
sub_df = pd.read_csv(SUBMISSION_FILE)
print(f"Shape: {sub_df.shape}")
print(sub_df.to_string(index=False))
print(f"\nReasoning stats:")
print(f"  Unique : {sub_df['reasoning'].nunique()} / 100")
print(f"  Avg len: {sub_df['reasoning'].str.len().mean():.0f} chars")
print(f"  Min len: {sub_df['reasoning'].str.len().min()} chars")
print(f"  Max len: {sub_df['reasoning'].str.len().max()} chars")
